In [ ]:
import geopandas as gpd


from benchmark_pipeline.generator.template_question.multihop.generator_multihop import make_multihop_dataset
from benchmark_pipeline.path import DATA


In [ ]:
def load_tables(repo):
    df_question = gpd.read_parquet(DATA / repo / "tables" / "df_question.parquet")
    df_street = gpd.read_parquet(DATA / repo / "tables" / "df_street.parquet")
    df_area = gpd.read_parquet(DATA / repo / "tables" / "df_area.parquet")
    df_osm = gpd.read_parquet(DATA / repo / "tables" / "df_osm.parquet")
    return df_question, df_osm, df_street, df_area

In [ ]:
template = {15:0, 14:1, 13:2, 12:3, 11:2, 10:4, 9:2, 8:3, 7:1, 6:2, 5: 2, 4: 3, 3: 3, 2: 3, 1: 2}
nb_start = 20
repo = "benchmark_fake_city_0"

In [ ]:
make_multihop_dataset(    
    nb_start,
    repo,
    template,
    *load_tables(repo)
    )

In [6]:
def sample_by_depth(df_multihop, k, seed=0):
    """k random questions per depth, all of them when a depth has fewer.

    Args:
        df_multihop (DataFrame): multihop questions, `template` as a dict or a list.
        k (int): questions kept per depth.
        seed (int): random seed.

    Returns:
        DataFrame: `query_texte` and `answers`, sorted by depth.
    """
    # template has one entry per node, plus a closing 0
    depth = df_multihop["template"].map(len) - 1
    shuffled_depth = depth.sample(frac=1, random_state=seed)
    kept_depth = shuffled_depth.groupby(shuffled_depth).head(k).sort_values(kind="stable")
    dump = df_multihop.loc[kept_depth.index, ["description_textuelle", "answers"]]
    dump = dump.rename(columns={"description_textuelle": "query"}).assign(depth=kept_depth)
    return dump

In [7]:
size_by_depth = 10
sample_by_depth(df_multihop, size_by_depth).to_parquet(multihop_dir / f"df_multihop_sample_{size_by_depth}.parquet")